### Data Ingestion

In [1]:
### Document datastrucutre

from langchain_core.documents import Document

In [2]:
doc = Document(
    page_content="This is the main text content I am using to create RAG",
    metadata = {
        "source":"example.txt",
        "pages":1,
        "author":"Tushar Singh",
        "date_created":"2025-01-01"
        
    }
)

doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Tushar Singh', 'date_created': '2025-01-01'}, page_content='This is the main text content I am using to create RAG')

In [3]:
### Create a simple txt file

import os
os.makedirs("../data/text_files" , exist_ok = True)

In [4]:
sample_texts = {
    "../data/text_files/python_intro.txt":"""python programming Introduction
    
    Python is a popular programming language. It was created by Guido van Rossum, and released in 1991.

It is used for:

web development (server-side),
software development,
mathematics,
system scripting.
What can Python do?
Python can be used on a server to create web applications.
Python can be used alongside software to create workflows.
Python can connect to database systems. It can also read and modify files.
Python can be used to handle big data and perform complex mathematics.
Python can be used for rapid prototyping, or for production-ready software development.""",

    "../data/text_files/machine_learning.txt":"""Machine Learning Introduction
    Machine Learning is a technique that allows computers to learn from data and make decisions without explicit programming. It works by identifying patterns in data and using them to make predictions. It is used in areas such as:

Image Recognition
Speech Processing
Language Translation
Recommender Systems
why_machine_learning_matters.webp
Need for Machine Learning
Machine Learning is important because traditional programming cannot handle complex tasks or large amounts of data efficiently. ML overcomes this by learning from data and making predictions without fixed rules. It is needed for the following reasons:

1. Solving Complex Business Problems
Traditional programming struggles with tasks like language understanding and medical diagnosis. ML learns from data and predicts outcomes easily.

Examples:

Image and speech recognition in healthcare.
Language translation and sentiment analysis.
2. Handling Large Volumes of Data
The internet generates huge amounts of data every day. Machine Learning processes and analyzes this data quickly by providing valuable insights and real time predictions.

Examples:

Fraud detection in financial transactions.
Personalized feed recommendations on Facebook and Instagram from billions of interactions."""
}

In [5]:
for filepath , content in sample_texts.items():
    with open(filepath , 'w' , encoding='utf-8') as f:
        f.write(content)
        
print("Sample text file created")
    

Sample text file created


In [6]:
###Text Loader

from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt" , encoding="utf-8")
document = loader.load()
print(document)

c:\Desktop\RAG-Krish Nayak\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='python programming Introduction\n    \n    Python is a popular programming language. It was created by Guido van Rossum, and released in 1991.\n\nIt is used for:\n\nweb development (server-side),\nsoftware development,\nmathematics,\nsystem scripting.\nWhat can Python do?\nPython can be used on a server to create web applications.\nPython can be used alongside software to create workflows.\nPython can connect to database systems. It can also read and modify files.\nPython can be used to handle big data and perform complex mathematics.\nPython can be used for rapid prototyping, or for production-ready software development.')]


In [7]:
### Directory Loader

from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/text_files",
    glob= "**/*.txt",
    loader_cls= TextLoader,
    loader_kwargs={"encoding" :"utf-8"},
    show_progress=False
)

documents = dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Introduction\n    Machine Learning is a technique that allows computers to learn from data and make decisions without explicit programming. It works by identifying patterns in data and using them to make predictions. It is used in areas such as:\n\nImage Recognition\nSpeech Processing\nLanguage Translation\nRecommender Systems\nwhy_machine_learning_matters.webp\nNeed for Machine Learning\nMachine Learning is important because traditional programming cannot handle complex tasks or large amounts of data efficiently. ML overcomes this by learning from data and making predictions without fixed rules. It is needed for the following reasons:\n\n1. Solving Complex Business Problems\nTraditional programming struggles with tasks like language understanding and medical diagnosis. ML learns from data and predicts outcomes easily.\n\nExamples:\n\nImage and speech recognition in healthcare.\n

In [8]:
### PDF Loader

from langchain_community.document_loaders import PyPDFLoader ,PyMuPDFLoader ,DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/pdf",
    glob= "**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False
)

pdf_documents= dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\attention.pdf', 'file_path': '..\\data\\pdf\\attention.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'trapped': '', 'modDate': "D:20180212212210-08'00'", 'creationDate': '', 'page': 0}, page_content='Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Pol

In [9]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [10]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: attention.pdf
  ✓ Loaded 11 pages

Processing: LLM pdf.pdf
  ✓ Loaded 30 pages

Total documents loaded: 41


In [11]:
all_pdf_documents

[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On 

In [12]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [13]:
chunks=split_documents(all_pdf_documents)
chunks

Split 41 documents into 167 chunks

Example chunk:
Content: Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz...
Metadata: {'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiori

[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On 

### Embedding and Vector DB

In [14]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List , Dict , Any , Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [15]:
class EmbeddingManager:
    """Handles document embedding genration using SentenceTransformer"""
    def __init__(self , model_name : str ="all-MiniLM-L6-v2"):
        """_summary_

        Args:
            model_name (str, optional): _description_. Defaults to "all-MiniLM-L6-v2".
        """
        
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """Load the Sentence Transformer Model"""
        
        try:
            print(f"Loading embedding model : {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded sucessfully. Embedding Dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name} : {e}")
            raise
        
    def genrate_embeddings(self , texts:List[str])->np.ndarray:
        """_summary_

        Args:
            texts (List[str]): _description_

        Returns:
            np.ndarray: _description_
        """
        
        if not self.model:
            raise ValueError("Model Not loaded")
        
        print(f"genrating embedding for {len(texts)} texts...")
        embeddings =self.model.encode(texts , show_progress_bar=True)
        print(f"genrated embedding with shape {embeddings.shape}")
        
        return embeddings
    
    
    
### initialize the embedding manager

embedding_manager = EmbeddingManager()

embedding_manager
    

Loading embedding model : all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3728.72it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded sucessfully. Embedding Dimension: 384


### Vector Store

In [16]:
class VectorStore:
    """Manage Document Embeddings in a chromaDB vector store"""
    
    def __init__(self , collection_name : str = "pdf_documents" , persist_directory : str = "../data/vector_store"):
        """_summary_

        Args:
            collection_name (str, optional): _description_. Defaults to "pdf_documents".
            persist_directory (str, optional): _description_. Defaults to "../data/vector_store".
        """
        
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        """_summary_
        Initialize chromadb client and collection
        """
        try:
            # create persistant chromadb client
            os.makedirs(self.persist_directory , exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # get or create collections with cosine distance metric
            # Delete existing collection to recreate with correct metric
            try:
                self.client.delete_collection(name=self.collection_name)
            except:
                pass
            
            self.collection = self.client.create_collection(
                name = self.collection_name,
                metadata={"hnsw:space": "cosine", "description" : "pdf documents embedding for RAG"}
            )
            print(f"vector store initialized . collection : {self.collection_name}")
            print(f"Existing documents in collection : {self.collection.count()}")
            
        except Exception as e:
            print(f"Error Initialize in vector store : {e}")
            raise
            
            
    def add_documents(self , documents:List[Any] , embeddings : np.ndarray):
        """_summary_
        
        Add documents and their embeddings to the vector store

        Args:
            documents (List[Any]): List of langchain documents
            embeddings (np.ndarray): crossponding embeddings for the documents
        """
        
        if len(documents) != len(embeddings):
            raise ValueError ("Number of document must match number of embedding")
        
        print(f"Adding {len(documents)} documents to vector store..")
        
        # prepare data for chromadb
        
        ids = []
        metadatas = []
        documents_text = []
        embedding_list = []
        
        for i, (doc , embedding) in enumerate(zip(documents , embeddings)):
            # genrate unique id
            
            doc_id = (f"doc_{uuid.uuid4().hex[:8]}_{i}")
            ids.append(doc_id)
            
            # prepare metadata
            
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)            
            metadatas.append(metadata)
            
            # document content
            
            documents_text.append(doc.page_content)
            
            # embedding
            
            embedding_list.append(embedding.tolist())
            
        try:
            self.collection.add(
                ids=ids,
                embeddings= embedding_list,
                metadatas= metadatas,
                documents= documents_text
            )
            
            print(f"successfully loaded {len(documents)} documnets to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"error adding document to vector store {e}")
            raise
        
        
        
vectorstore = VectorStore()
vectorstore
                
        

vector store initialized . collection : pdf_documents
Existing documents in collection : 0


In [17]:
chunks

[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On 

In [18]:
# Convert the text to embeddings 

texts = [doc.page_content for doc in chunks]


# Genrate the Embeddings

embeddings = embedding_manager.genrate_embeddings(texts)

# Store into the vector DataBase

vectorstore.add_documents(chunks , embeddings)

genrating embedding for 167 texts...


Batches: 100%|██████████| 6/6 [00:14<00:00,  2.36s/it]


genrated embedding with shape (167, 384)
Adding 167 documents to vector store..
successfully loaded 167 documnets to vector store
Total documents in collection: 167


### Retriever Pipeline from VectorStore

In [19]:
class RAGRetriever:
    """Handle query based retrieval from the vector store"""
    
    def __init__(self , vector_store:VectorStore , embeddings_manager : EmbeddingManager ):
        """
        Initialize the Retriever
        
        
        Args:
            vector_store : vectorstore containig documents embedding
            embedding_manager : manager for genrating query embeddings
        """
        
        self.vector_store = vector_store
        self.embeddings_manager = embeddings_manager
        
        
    def retrieve(self , query : str , top_k : int = 5 , score_threshold : float = -1.0) -> List[Dict[str , Any]]:
        """
        Retrieve relevent documents form a query
        
        
        Args:
            query : the search query
            top_k : Number of top result to return
            score_thresold : Minimum similarity score thresold (default -1.0 accepts all results)
            
        Returns: 
             
            List of dictionaries containg reterived documents and metadata
        """
        
        print(f"Retriving documents for query: {query}")
        print(f"Top K {top_k} , Score thresold : {score_threshold}")
        
        
        # Genrate query embeddings
        
        query_embedding = self.embeddings_manager.genrate_embeddings([query])[0]
        
        # Search in the Vector store
        
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results= top_k
            ) 
            
            # Process result
            
            retrieved_doc = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                
                for i , (doc_id , document , metadata , distance ) in enumerate(zip(ids , documents , metadatas , distances)):
                    # Convert L2 Euclidean distance to cosine similarity for normalized embeddings
                    # For unit vectors: euclidean_distance^2 = 2 * (1 - cosine_similarity)
                    # So: cosine_similarity = 1 - (euclidean_distance^2 / 2)
                    similarity_score = 1 - (distance ** 2 / 2)
                    
                    
                    if similarity_score >= score_threshold:
                        retrieved_doc.append({
                            'id' : doc_id,
                            'content' : document,
                            'metadata' : metadata, 
                            'similarity_score' : similarity_score,
                            'distance' : distance,
                            'rank' : i + 1
                        })
                print(f"Retrieved {len(retrieved_doc)} documents (after filtering)")      
            else:
                print("No document found")
                
                
            return retrieved_doc
                
        except Exception as e:
            print(f"Error during retrievel : {e}")
            return []
        
        
rag_retriever = RAGRetriever(vectorstore , embedding_manager)   

rag_retriever   

In [20]:
rag_retriever.retrieve("what is llm")

Retriving documents for query: what is llm
Top K 5 , Score thresold : -1.0
genrating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.12it/s]

genrated embedding with shape (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_cd7bc122_134',
  'content': '24 CHAPTER 7 • L ARGE LANGUAGE MODELS\nthat people tend to assign human characteristics to computers and interact with them\nin ways that are typical of human-human interactions. They interpret an utterance in\nthe way they would if it had spoken by a human, (even though they are aware they\nare talking to a computer). Thus LLMs have had signiﬁcant inﬂuences on people’s\ncognitive and emotional state, leading to problems like emotional dependence on\nLLMs. These issues (emotional engagement and privacy) mean we need to think\ncarefully about the impact of LLMs on the people who are interacting with them.\nIn addition to their ability to harm their users in these ways, LLMs may carry out\nadditional harmful activities themselves, especially as agent-based paradigms makes\nit possible for language models to directly interact with the world.\nLanguage models can also be used by malicious actors for generating text for\nfraud, phishing, propaganda,

### Integration vectorDB context pipeline with LLM Output

In [21]:
# Simple RAG pipeline with GROQ LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv(".env")

#initlialize the groq llm 

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    groq_api_key = groq_api_key ,
    model_name ="openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=1024
)

# Simple RAG Function -> Retrieve context + genrate answer

def rag_simple(query , retriever , llm , top_k=3):
    #retrieve the context
    results = retriever.retrieve(query , top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results])if results else ""
    if not context:
        return "No relevent context found to the answer question"
    
    
    # genrate the answer using groq llm
    
    prompt = f"""Use this following context to answer the question concisely .
        Context:
        {context}
        
        Question : {query}
        
        Answer : 
        
        """
    response = llm.invoke([prompt.format(context = context , question = query)])
    return response.content

In [22]:
answer = rag_simple("what is attention mechanism ?" , rag_retriever , llm)
print(answer)

Retriving documents for query: what is attention mechanism ?
Top K 3 , Score thresold : -1.0
genrating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.86it/s]


genrated embedding with shape (1, 384)
Retrieved 3 documents (after filtering)
**Attention mechanism** is a neural network component that lets a model weigh the importance of different positions in a sequence when computing a representation for a target position. In practice, it computes a weighted sum of “value” vectors, where the weights (attention scores) are derived from the similarity between “query” and “key” vectors. This allows the model to focus on relevant parts of the input (or output) in parallel, enabling efficient, interpretable, and context‑aware processing.


## Enhanced RAG Pipeline Features

In [23]:
def rag_advanced(query , retriever , llm , top_k=5 , min_score=0.1 , return_context=False):
    """ 
    RAG Pipeline with extra features:
    - Returns answer source confidence, score and optionally ful context
    
    """
    
    results = retriever.retrieve(query , top_k=top_k , score_threshold = min_score)
    if not results:
        return {'answer' : 'No relevent context found.' , 'sources' :[] , 'confidence' : 0.0 , "context" : ''}
    
    context = "\n\n.".join([doc['content'] for doc in results])
    sources = [{
        'source' : doc['metadata'].get('source_file' ,doc['metadata'].get('source' , 'unknown')),
        'page' : doc['metadata'].get('page' , 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] +'...'
    }for doc in results]
    confidence = max([doc['similarity_score']for doc in results])
    
    prompt = f"""Use this following context to answer the question concisely .
        Context:
        {context}
        
        Question : {query}
        
        Answer : 
        
        """
    response = llm.invoke([prompt.format(context = context , question = query)])
    output = {
        'answer' : response.content,
        'sources' : sources,
        'confidence' : confidence
    }
    if return_context:
        output['context'] = context
        
    return output

In [24]:
result = rag_advanced("Three architectures for language models?" , rag_retriever , llm , top_k= 5 , min_score = 0.1 , return_context = True)
print("Answer : " , result['answer'])
print("Sources : " , result['sources'] )
print("Confidence : " , result['confidence'] )
print("Context preview : " , result['context'][:300])


Retriving documents for query: Three architectures for language models?
Top K 5 , Score thresold : 0.1
genrating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.25it/s]

genrated embedding with shape (1, 384)
Retrieved 5 documents (after filtering)


Answer :  **Three architectures for language models**

1. **Encoder** – processes input tokens and produces vector representations.  
2. **Decoder** – generates output tokens one at a time, conditioned on previous tokens.  
3. **Encoder‑Decoder** – combines both: encodes input tokens and then decodes to generate a sequence of output tokens.
Sources :  [{'source': 'LLM pdf.pdf', 'page': 3, 'score': 0.9247870261988265, 'preview': 'model, which is the language model architecture we will deﬁne in this chapter, is\nactually only one of three common LM architectures.\nThe three architectures are the encoder, the decoder, and the encoder-decoder.\nFig. 7.3 gives a schematic picture of the three.\nw w w\nw w w\nw w w w w\nw w w w w\nw w w ...'}, {'source': 'LLM pdf.pdf', 'page': 23, 'score': 0.9162117838625292, 'preview': 'conditionally generate text.\n• There are three major architectures for language models: the encoder, the\ndecoder, and the encoder-decoder. The well-known large language mo

In [25]:
#----Advanced RAG Pipeline : streaming , citations , History , summarization

from typing import List , Dict , Any
import time

class AdvancedRAGPipeline:
    def __init__(self , retriever , llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  #store query history
        
        
    def query (self , question:str ,top_k: int=5 , min_score: float=0.2 , stream : bool = False , summarize : bool = False):
        # Retrieve relevent documents
        
        results =self.retriever.retrieve(question , top_k = top_k , score_threshold = min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content']for doc in results])
            sources = [{
                'source':doc['metadata'].get('source_file' , doc['metadata'].get('source' , 'unknown')),
                'page' : doc['metadata'].get('page' , 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] +'...'
            }for doc in results]
        
            # streaming answer simulation
            
            prompt = f"""Use this following context to answer the question concisely .
                Context:
                {context}
                
                Question : {question}
                
                Answer : 
                
                """
            if stream : 
                print("Streaming answer : ")
                for i in range(0 , len(prompt) , 80):
                    print(prompt[i:i+80] , end="" , flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context , question = question)])
            answer = response.content
            
        # Add citations to answer
        
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i , src in enumerate(sources)]
        answer_with_citations = (answer + "\n\n Citations: \n" + "\n".join(citations)) if citations else answer
        
        
        # Optionally summarize answer
        
        summary = None
        
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences : \n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content
            
        # store query history
        self.history.append({
            "question" : question,
            'answer' : answer,
            'sources' : sources,
            'summary' : summary
        })
        
        return {
            'question': question,
            'answer' : answer_with_citations,
            'sources' : sources,
            'summary' : summary,
            "history" : self.history 
        }
          
# Example usages

adv_rag = AdvancedRAGPipeline(rag_retriever , llm)  
result = adv_rag.query("Who is Tushar Singh" , top_k=3 , min_score=0.3 , stream=True , summarize=True)
print("\n Final Answer" , result['answer'])
print("Summary" , result['summary'])
print("History" , result['history'][-1])


        

Retriving documents for query: Who is Tushar Singh
Top K 3 , Score thresold : 0.3
genrating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.00it/s]

genrated embedding with shape (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer : 
Use this following context to answer the question concisely .
                Context:
                tensorflow/tensor2tensor.
Acknowledgements We are grateful to Nal Kalchbrenner and Stephan Gouws for their fruitful
comments, corrections

 and inspiration.
9

can also be an instruction (like “ Translate the following sentence into
Hindi: ‘Chop the garlic finely’ ”).
More explicit prompts that specify the set of possible answers lead to better
performance. For example, here is a prompt template to do sentiment analysis that
prespeciﬁes the potential answers:
A prompt consisting of a review plus an incomplete statement
Human: Do you think that “input” has negative or positive sentiment?
Choices:
(P) Positive
(N) Negative
Assistant: I believe the best answer is: (
This prompt uses a number of more sophisticated prompting characteristics. It
speciﬁes the two allowable choices (P) and (N), and ends the prompt with the open
parenthesis that strongly suggests the answer will be (P) or (N). Note that it also
speciﬁes the role of the language model as an assistant.
Including some labeled examples in the prompt can also improve performance.
We call such examples demonstrations. The task of prompting with examplesdemonstrations

J